# Alkaline platform 270

Every cell here moves a real arm. Two things that are true no matter what the code says:

- **The physical e-stop is the only reliable stop.** The voice stop in the RobotStop
  repo is measured at 42% recall; the software envelope below does not constrain
  joint-space moves at all.
- **`home()` is a joint-space move and nothing checks it.** Its angles were last set
  for a robot swap and drove the arm into the table on 2026-09-16. Raise the arm to a
  clear pose before calling it, and see the warning at `ang_dict` in
  `robotic_testing/common/robotic_arms/xarm7/xarm7_config.py`.

The same arm control is available as a command line tool, which needs no notebook
server: `python robotic_testing/vla.py --help`

## Session

The full alkaline platform: EC-Lab, the camera, TeamViewer/ToDesk prompt bypasses and
optionally the database. `rt.run(...)` below needs all of it. For arm-only work, use
`init_vla_platform` instead -- see `vla.ipynb`.

In [ ]:
from robotic_testing.platforms import init_alkaline_platform_270

exp_name = 'test'
rt = init_alkaline_platform_270(exp_name=exp_name, connect_db=False)
arm = rt.arm

In [ ]:
# start in n hours instead of now
import time

n = 0.1
time.sleep(3600 * n)

## Electrochemistry runs

Every `run_config` that used to have a cell of its own, kept as data. Nine cells that
differed only in their values are one table and one runner, so a new run is an entry
rather than another copy of the loop.

Nothing is lost in the collapse. `rt.run` defaults `immerse_option` to
`'flask_contact_immersed'`, which is exactly what the cells that commented that key out
were getting, and `data_analysis=False` was in every one of them.

`benchmark={30: 'Pt_electrode'}` was commented out everywhere; it is listed once below
as a reminder that the parameter exists.

In [ ]:
RUNS = {
    # name                    sequence  sample_id_list          rack  operator
    'zhen_rack15_seq':      dict(sequence=True,  sample_id_list=[1, 18],               sample_rack_id_starting=15, operator_name='zhen'),
    'zhen_rack3':           dict(sequence=False, sample_id_list=[4],                   sample_rack_id_starting=3,  operator_name='zhen'),
    'zhen_rack0_1001':      dict(sequence=False, sample_id_list=[1001],                sample_rack_id_starting=0,  operator_name='zhen'),
    'zhen_rack18_seq':      dict(sequence=True,  sample_id_list=[107, 120],            sample_rack_id_starting=18, operator_name='zhen'),  # from 91; sample 109 damaged
    'zhen_rack32':          dict(sequence=False, sample_id_list=['59'],                sample_rack_id_starting=32, operator_name='zhen'),
    'weiyin_rack0_steel':   dict(sequence=False, sample_id_list=['low_carbon_steel6'], sample_rack_id_starting=0,  operator_name='weiyin'),
    'weiyin_rack0_1_3':     dict(sequence=False, sample_id_list=[1, 3],                sample_rack_id_starting=0,  operator_name='Weiyin'),
    'dohun_rack7_reccell':  dict(sequence=False, sample_id_list=['Rec_Cell4_2'],       sample_rack_id_starting=7,  operator_name='dohun'),
    'dohun_rack2':          dict(sequence=False, sample_id_list=[1],                   sample_rack_id_starting=2,  operator_name='dohun'),
}

# Other parameters rt.run accepts, all optional:
#   immerse_option='flask_contact_immersed'   the default; '6cm_sample_XAS', 'small_cell', ...
#   benchmark={30: 'Pt_electrode'}            was commented out in every original cell
#   data_analysis=False                       the default

run = 'zhen_rack15_seq'

print(run, RUNS[run])
# add %%capture as the first line of this cell to suppress the run's output
# rt.run(data_analysis=False, **RUNS[run])

## Arm basics

In [ ]:
# Uncomment the one you want. Each is a complete move on its own.

arm.home()
# arm.move_to_mid_station()
# arm.open_gripper()
# arm.close_gripper()

# where is it now, and is that inside the safety envelope?
# print(arm.describe_state())
# print(arm.describe_envelope())

## Samples

In [ ]:
sample = 0

arm.load_sample(sample)
# arm.unload_sample(sample)

# arm.pick_up_sample(sample)
# arm.put_sample_back(sample)

# a rack position that needs the offset correction applied
# arm.pick_up_sample(11, correction=True)

# a whole rack, one at a time
# for i in range(30, 36):
#     arm.pick_up_sample(i)
#     arm.put_sample_back(i)

## Flask and rinsing

In [ ]:
arm.move_to_flask()

# in, then back out
# arm.sink_in_flask()
# arm.sink_in_flask(reverse=True)

# arm.rinsing()

## Immersion sweep with per-sample speeds

Loads each sample, dips it in and out at a chosen speed, and puts it back. Samples not
listed in `sample_speeds` use `DEFAULT_SPEED`.

`sink_in_flask_controlled_speed(reverse=False, immerse_option='flask_contact_immersed',
speed=0.5)` -- so `speed` is the only argument worth varying here, and the default is
0.5 rather than the 1 used below.

In [ ]:
# [speed in, speed out] per sample; anything not listed uses DEFAULT_SPEED
DEFAULT_SPEED = 1
sample_speeds = {
    0: [1, 1],
}

for sample in range(0, 30):
    arm.load_sample(sample)

    speed_in, speed_out = sample_speeds.get(sample, [DEFAULT_SPEED, DEFAULT_SPEED])
    arm.sink_in_flask_controlled_speed(reverse=False, speed=speed_in)
    arm.sink_in_flask_controlled_speed(reverse=True, speed=speed_out)

    arm.unload_sample(sample)

## Free-form motion

Held to the envelope in the arm's config. A move outside it, or bigger than one step, is
refused before anything moves -- `dry_run=True` reports where a move would end up without
going there.

The envelope is a box with a keep-out cylinder around the base. It constrains
`move_relative` and friends; it does **not** constrain `home()` or any other joint-space
move.

In [ ]:
from robotic_testing.common.robotic_arms.xarm_control import SafetyError

# where would this end up?
print('predicted:', arm.move_relative(dx=30, dy=-20, dz=50, dry_run=True))

# refused before it starts
try:
    arm.move_relative(dz=5000)
except SafetyError as error:
    print('refused: ', error)

# actually move
# arm.move_relative(dx=30, dy=-20, dz=50)
# arm.move_up(40)
# arm.move_down(40)
# arm.move_to_pose(z=300)          # absolute; None keeps the current value
# arm.move_relative_path([[0, 0, 80], [60, 0, 0], [0, 0, -80]], dry_run=True)

## VLA section

This was a hand-written copy of the planner, the pydantic action vocabulary and the
executor. All three now live in `robotic_testing/vla.py` and `common/vla_control.py`,
which is what the command line tool runs and where the envelope checks, the model
fallback chain and the tests are. Imported here rather than maintained twice.

In [ ]:
from robotic_testing.common.vla_control import VLARobot
from robotic_testing.vla import make_planner, handle_instruction, run_plan

vla = VLARobot(arm, logger=getattr(rt, 'logger', None))
planner = make_planner(vla)          # default model chain; make_planner(vla, 'gemini-3.8-flash') to pick one

print(vla.system_prompt()[:400], '...')

In [ ]:
# dry_run=True plans and validates without moving
handle_instruction(vla, planner, 'wave at me', dry_run=True, assume_yes=False)

# handle_instruction(vla, planner, 'lift straight up by 4 cm, then go home', dry_run=True, assume_yes=False)
# handle_instruction(vla, planner, 'grab the beaker on the far side of the bench', dry_run=True, assume_yes=False)

# for real, with a confirmation prompt before anything moves
# handle_instruction(vla, planner, 'wave at me three times', dry_run=False, assume_yes=False)

# a plan written by hand, no planner involved
# run_plan(vla, [{'action': 'wave', 'times': 2}], dry_run=True, assume_yes=False)

## Stopping, and recovering

`emergency_stop()` puts the controller into the stop state, abandoning the current move.
`reset_safety_state()` clears the fault and re-enables motion -- only once the cause is
understood.

In [ ]:
# arm.emergency_stop()
# arm.reset_safety_state()